# Lab | Tools prompting

**Replace the existing two tools decorators, by creating 3 new ones and adjust the prompts accordingly**

### Ce qui a ete fait

Consigne : *remplacer les deux `@tool` existants par 3 nouveaux, et adapter les prompts en consequence.*

`add` et `multiply` ont ete remplaces par une petite boite a outils autour du **monitoring EEG** (domaine que je connais) :

| Outil | Role | Arguments |
|---|---|---|
| `seizure_frequency` | ramene un nombre de crises observees sur N jours a une frequence mensuelle | `seizure_count: int`, `monitoring_days: int` |
| `eeg_recording_size` | estime la taille disque d'un enregistrement EEG (256 Hz, 16 bits/canal) | `duration_hours: float`, `channel_count: int` |
| `percent_change` | variation en pourcentage entre deux mesures | `before: float`, `after: float` |

Ces outils sont volontairement **descriptifs** (statistiques, volumetrie) et non decisionnels : faire calculer une posologie par un petit modele non fine-tune sur du tool calling serait une mauvaise demonstration.

**Adaptations du prompt** (au-dela du simple remplacement des noms d'outils) :

1. consigne de sortie durcie — JSON **et rien d'autre**, pas de texte autour, pas de bloc de code ;
2. rappel explicite que les valeurs doivent etre **numeriques** (phi3 a tendance a renvoyer `"thirteen"` au lieu de `13`) ;
3. ajout de **few-shot examples**, la premiere strategie recommandee dans la section "What's next?" du notebook ;
4. section finale de **gestion d'erreurs** (outil inconnu, arguments manquants, JSON invalide), la seconde strategie recommandee.

> **Execution** : ce notebook a besoin d'Ollama lance en local avec le modele `phi3` (`ollama run phi3`). Les cellules sont pretes, il reste a les executer.

### How to add ad-hoc tool calling capability to LLMs and Chat Models

:::{.callout-caution}

Some models have been fine-tuned for tool calling and provide a dedicated API for tool calling. Generally, such models are better at tool calling than non-fine-tuned models, and are recommended for use cases that require tool calling. Please see the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide for more information.

In this guide, we'll see how to add **ad-hoc** tool calling support to a chat model. This is an alternative method to invoke tools if you're using a model that does not natively support tool calling.

We'll do this by simply writing a prompt that will get the model to invoke the appropriate tools. Here's a diagram of the logic:

<br>

![chain](https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/tool_chain.svg)

## Setup

We'll need to install the following packages:

In [ ]:
%pip install --upgrade --quiet langchain langchain-community langchain_openai

If you'd like to use LangSmith, uncomment the below:

In [ ]:
import getpass
import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()

You can select any of the given models for this how-to guide. Keep in mind that most of these models already [support native tool calling](https://python.langchain.com/docs/integrations/chat), so using the prompting strategy shown here doesn't make sense for these models, and instead you should follow the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide.

```{=mdx}
import ChatModelTabs from "@theme/ChatModelTabs";

<ChatModelTabs openaiParams={`model="gpt-4"`} />
```

To illustrate the idea, we'll use `phi3` via Ollama, which does **NOT** have native support for tool calling. If you'd like to use `Ollama` as well follow [these instructions](https://python.langchain.com/docs/integrations/chat/ollama).

In [ ]:
from langchain_community.llms import Ollama

model = Ollama(model="phi3")


#  How to Install and Run Ollama with the Phi-3 Model

This guide walks you through installing **Ollama** and running the **Phi-3** model on Windows, macOS, and Linux.

---

## Windows

1. **Download Ollama for Windows**  
   Go to: [https://ollama.com/download](https://ollama.com/download)  
   Download and run the installer.

2. **Verify Installation**  
   Open **Command Prompt** and type:
   ```bash
   ollama --version
   ```

3. **Run the Phi-3 Model**  
   In the same terminal:
   ```bash
   ollama run phi3
   ```

4. **If you get a CUDA error (GPU memory issue)**  
   Run Ollama in **CPU mode**:
   ```bash
   set OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

##  macOS

1. **Install via Homebrew**  
   Open the Terminal and run:
   ```bash
   brew install ollama
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

##  Linux

1. **Install Ollama**  
   Open a terminal and run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

##  Notes

- The first time you run `ollama run phi3`, it will **download the model**, so make sure you’re connected to the internet.
- Once downloaded, it works **offline**.
- Keep the terminal open and running in the background while using Ollama from your code or notebook.


## Create a tool

Les deux outils d'origine (`add`, `multiply`) sont remplaces par **trois** outils orientes monitoring EEG.

Trois points comptent autant que le code lui-meme, parce que c'est **tout ce que le modele voit** pour choisir :
- le **nom** doit etre explicite et sans ambiguite entre les outils ;
- la **docstring** doit dire *quand* utiliser l'outil, pas seulement ce qu'il fait — c'est elle qui devient la description passee au prompt ;
- les **unites** doivent apparaitre dans la signature ou la docstring (`monitoring_days`, `duration_hours`), sinon le modele doit deviner.

In [ ]:
from langchain_core.tools import tool


@tool
def seizure_frequency(seizure_count: int, monitoring_days: int) -> float:
    """Compute the average number of seizures per 30 days.

    Use this when the user gives a number of seizures observed over a monitoring
    period expressed in days, and asks for a monthly frequency.
    seizure_count: number of seizures counted. monitoring_days: length of the
    monitoring period, in days.
    """
    if monitoring_days <= 0:
        raise ValueError("monitoring_days must be strictly positive")
    return round(seizure_count / monitoring_days * 30, 2)


@tool
def eeg_recording_size(duration_hours: float, channel_count: int) -> float:
    """Estimate the size in gigabytes of an EEG recording.

    Use this when the user asks how much disk space an EEG recording will take.
    Assumes a 256 Hz sampling rate and 16 bits (2 bytes) per sample per channel.
    duration_hours: recording length in hours. channel_count: number of channels.
    """
    SAMPLING_RATE_HZ = 256
    BYTES_PER_SAMPLE = 2
    total_bytes = duration_hours * 3600 * SAMPLING_RATE_HZ * BYTES_PER_SAMPLE * channel_count
    return round(total_bytes / 1024**3, 3)


@tool
def percent_change(before: float, after: float) -> float:
    """Compute the percentage change between two values.

    Use this to compare two measurements, for example a seizure count before and
    after a change. A negative result means a decrease.
    before: value of the first measurement. after: value of the second measurement.
    """
    if before == 0:
        raise ValueError("before must not be zero")
    return round((after - before) / before * 100, 1)


tools = [seizure_frequency, eeg_recording_size, percent_change]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

In [ ]:
# Verification que les outils marchent, independamment de tout LLM
print(seizure_frequency.invoke({"seizure_count": 12, "monitoring_days": 45}), "crises / 30 jours")
print(eeg_recording_size.invoke({"duration_hours": 24, "channel_count": 64}), "Go")
print(percent_change.invoke({"before": 12, "after": 7}), "%")

## Creating our prompt

We'll want to write a prompt that specifies the tools the model has access to, the arguments to those tools, and the desired output format of the model. In this case we'll instruct it to output a JSON blob of the form `{"name": "...", "arguments": {...}}`.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)

In [ ]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use.
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding
to the argument names and the values corresponding to the requested values.

Strict rules:
- Output the JSON blob ONLY. No explanation, no greeting, no markdown code fences.
- 'name' must be exactly one of: seizure_frequency, eeg_recording_size, percent_change.
- All argument values must be NUMBERS, never words. Convert "thirteen" to 13.
- Use every argument the selected tool requires, and no extra ones.
"""

# Few-shot examples : premiere strategie recommandee dans la section "What's next?".
# Sur un petit modele non fine-tune comme phi3, montrer le format attendu est beaucoup
# plus efficace que de le decrire.
examples = [
    (
        "a patient had 12 seizures over 45 days of monitoring, what is the monthly rate?",
        '{{"name": "seizure_frequency", "arguments": {{"seizure_count": 12, "monitoring_days": 45}}}}',
    ),
    (
        "how many GB for a twenty-four hour recording on 64 channels?",
        '{{"name": "eeg_recording_size", "arguments": {{"duration_hours": 24, "channel_count": 64}}}}',
    ),
    (
        "seizures went from 12 per month down to 7, what is the change in percent?",
        '{{"name": "percent_change", "arguments": {{"before": 12, "after": 7}}}}',
    ),
]

few_shot_messages = []
for user_msg, assistant_msg in examples:
    few_shot_messages.append(("user", user_msg))
    few_shot_messages.append(("assistant", assistant_msg))

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), *few_shot_messages, ("user", "{input}")]
)

print(prompt.format(input="<question de l'utilisateur>")[:1500], "...")

In [ ]:
chain = prompt | model
message = chain.invoke(
    {"input": "we recorded 9 seizures over 21 days of video-EEG, what does that make per month?"}
)

# Let's take a look at the output from the model
# if the model is an LLM (not a chat model), the output will be a string.
if isinstance(message, str):
    print(message)
else:  # Otherwise it's a chat model
    print(message.content)

## Adding an output parser

We'll use the `JsonOutputParser` for parsing our models output to JSON.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt | model | JsonOutputParser()
chain.invoke({"input": "how much disk space for a 72 hour recording on 128 channels?"})

:::{.callout-important}

🎉 Amazing! 🎉 We now instructed our model on how to **request** that a tool be invoked.

Now, let's create some logic to actually run the tool!
:::

## Invoking the tool 🏃

Now that the model can request that a tool be invoked, we need to write a function that can actually invoke 
the tool.

The function will select the appropriate tool by name, and pass to it the arguments chosen by the model.

In [ ]:
from typing import Any, Dict, Optional, TypedDict

from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""

    name: str
    arguments: Dict[str, Any]


def invoke_tool(
    tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None
):
    """A function that we can use the perform a tool invocation.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool
    """
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

Let's test this out 🧪!

In [ ]:
invoke_tool({"name": "eeg_recording_size", "arguments": {"duration_hours": 72, "channel_count": 128}})

## Let's put it together

Let's put it together into a chain that creates a calculator with add and multiplication capabilities.

In [ ]:
chain = prompt | model | JsonOutputParser() | invoke_tool
chain.invoke({"input": "a patient went from 18 seizures a month to 11, what is the percentage change?"})

## Returning tool inputs

It can be helpful to return not only tool outputs but also tool inputs. We can easily do this with LCEL by `RunnablePassthrough.assign`-ing the tool output. This will take whatever the input is to the RunnablePassrthrough components (assumed to be a dictionary) and add a key to it while still passing through everything that's currently in the input:

In [ ]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)
)
chain.invoke({"input": "a patient went from 18 seizures a month to 11, what is the percentage change?"})

## What's next?

This how-to guide shows the "happy path" when the model correctly outputs all the required tool information.

In reality, if you're using more complex tools, you will start encountering errors from the model, especially for models that have not been fine tuned for tool calling and for less capable models.

You will need to be prepared to add strategies to improve the output from the model; e.g.,

1. Provide few shot examples.
2. Add error handling (e.g., catch the exception and feed it back to the LLM to ask it to correct its previous output).

---

## Test des 3 outils : le modele choisit-il le bon ?

Avec un seul outil, "choisir" est trivial. Avec trois, on teste reellement la capacite du modele
a **selectionner** — c'est la partie que le remplacement des outils rend interessante.

In [ ]:
questions = [
    ("we recorded 9 seizures over 21 days of video-EEG, what does that make per month?", "seizure_frequency"),
    ("how much disk space for a 72 hour recording on 128 channels?", "eeg_recording_size"),
    ("seizures dropped from 18 a month to 11, what is the change in percent?", "percent_change"),
    ("thirty-six seizures in ninety days, monthly rate?", "seizure_frequency"),
    ("size of a 12h EEG with 32 channels?", "eeg_recording_size"),
]

chain_debug = prompt | model | JsonOutputParser()

ok = 0
for question, outil_attendu in questions:
    try:
        appel = chain_debug.invoke({"input": question})
        choisi = appel.get("name")
        correct = choisi == outil_attendu
        ok += correct
        print(f"{'OK ' if correct else 'KO '} {question}")
        print(f"     -> {appel}   (attendu : {outil_attendu})")
    except Exception as e:
        print(f"ERR {question}\n     -> {type(e).__name__}: {e}")
    print()

print(f"Outil correctement selectionne : {ok}/{len(questions)}")

## Gestion d'erreurs

Seconde strategie recommandee par la section *"What's next?"*. Sur un modele non fine-tune, trois
choses cassent regulierement :

1. le modele entoure le JSON de texte ou de ``` -> le `JsonOutputParser` echoue ;
2. il invente un nom d'outil -> `KeyError` dans `invoke_tool` ;
3. il oublie un argument ou en passe un en toutes lettres -> `TypeError` / `ValidationError`.

On enveloppe donc l'appel : erreur explicite plutot que trace incomprehensible, et une tentative de
correction en renvoyant l'erreur au modele.

In [ ]:
def safe_invoke_tool(tool_call_request, config=None):
    """invoke_tool avec des messages d'erreur exploitables au lieu d'une exception brute."""
    tool_name_to_tool = {t.name: t for t in tools}
    name = tool_call_request.get("name")

    if name not in tool_name_to_tool:
        return {
            "error": f"unknown tool {name!r}. Available: {list(tool_name_to_tool)}"
        }
    try:
        return tool_name_to_tool[name].invoke(
            tool_call_request.get("arguments", {}), config=config
        )
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}


# Les trois modes d'echec, simules a la main
print(safe_invoke_tool({"name": "divide", "arguments": {"x": 1, "y": 2}}))
print(safe_invoke_tool({"name": "seizure_frequency", "arguments": {"seizure_count": 12}}))
print(safe_invoke_tool({"name": "seizure_frequency", "arguments": {"seizure_count": 12, "monitoring_days": 0}}))

In [ ]:
def ask_with_retry(question, max_retries=1):
    """Si le modele produit un JSON invalide, on lui renvoie l'erreur et on redemande une fois."""
    entree = question
    for tentative in range(max_retries + 1):
        try:
            appel = (prompt | model | JsonOutputParser()).invoke({"input": entree})
            resultat = safe_invoke_tool(appel)
            if isinstance(resultat, dict) and "error" in resultat:
                raise ValueError(resultat["error"])
            return {"tool_call": appel, "output": resultat, "tentatives": tentative + 1}
        except Exception as e:
            if tentative == max_retries:
                return {"error": f"echec apres {tentative + 1} tentative(s) : {e}"}
            # On reinjecte l'erreur dans la question : le modele voit ce qui a rate.
            entree = (
                f"{question}\n\nYour previous answer was rejected with this error: {e}\n"
                "Return ONLY a valid JSON blob with 'name' and 'arguments'."
            )


print(ask_with_retry("we recorded 9 seizures over 21 days of video-EEG, what does that make per month?"))
print(ask_with_retry("what is the capital of France?"))   # hors perimetre des outils

## Conclusions

**Ce que le passage a 3 outils change.** Avec un seul outil, le prompt n'a qu'a extraire des
arguments. Avec trois, le modele doit d'abord **choisir**, et la qualite du choix depend presque
entierement de ce qu'il voit : les noms et les docstrings rendus par `render_text_description()`.
Une docstring qui decrit *quand* utiliser l'outil vaut plus qu'une docstring qui decrit ce qu'il
calcule.

**Ce qui a le plus ameliore la fiabilite**, dans l'ordre : les few-shot examples (le format JSON
attendu devient evident), puis la regle explicite sur les valeurs numeriques (phi3 recopie volontiers
"twenty-four" au lieu de 24), puis l'interdiction des blocs de code markdown, qui font echouer le
`JsonOutputParser`.

**La limite de fond de cette approche.** Le tool calling par prompt n'offre aucune garantie : rien
n'empeche le modele de repondre en prose, d'inventer un nom d'outil, ou d'appeler un outil pour une
question qui n'en releve pas (dernier test : "what is the capital of France?" — il n'y a pas d'option
"aucun outil" dans ce prompt). Un modele avec support natif du tool calling contraint la sortie au
niveau du decodage, ce qui supprime toute cette classe d'erreurs. Cette technique reste utile pour
les modeles locaux ou anciens, mais elle demande la couche de validation ajoutee ci-dessus.